In [1]:
"""
=============================================================================
Risk Guardrails for Counterfactual Explanations in AI Credit Scoring
=============================================================================

Paper: "Counterfactual Explanation (CFE)-Based Risk Guardrail Design
        for Adverse Selection Prevention and Consumer Protection
        in AI Credit Scoring Models"

Description:
    This notebook implements the full experimental pipeline for the paper,
    including XGBoost model training with Optuna optimization, counterfactual
    explanation generation using DiCE, and multi-scenario guardrail evaluation.

Repository Structure:
    Part 1 - Data Loading & Preprocessing
    Part 2 - XGBoost Model Training (Optuna Bayesian Optimization)
    Part 3 - Guardrail Configuration & Utility Functions
    Part 4 - Scenario Simulation (A / B / C)
    Part 5 - Sensitivity Analysis (Threshold ±10% ~ ±30%)
    Part 6 - Subgroup Analysis (Credit Score Quartiles Q1~Q4)
    Part 7 - Representative Case Extraction (Vanilla vs Proposed)
    Part 8 - Algorithm Robustness Validation (Random vs KD-Tree)

Requirements:
    pip install pandas numpy xgboost dice-ml optuna scikit-learn tqdm

Dataset:
    FICO HELOC (Home Equity Line of Credit) Dataset
    - Source: https://community.fico.com/s/explainable-machine-learning-challenge
    - Samples: 10,459 | Features: 23 | Target: RiskPerformance (Binary)

License: MIT
=============================================================================
"""

# %% [markdown]
# # Part 1: Data Loading & Preprocessing

# %%
import pandas as pd
import numpy as np
import xgboost as xgb
import dice_ml
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix)
from tqdm import tqdm
import warnings
import json
import time
from datetime import datetime

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Configuration ---
TOTAL_CFS = 4          # Number of counterfactual paths per sample
RANDOM_SEED = 42        # Global random seed for reproducibility
THRESHOLD = 0.20        # Actionability threshold (±20%)
TEST_SIZE = 0.2         # Train/test split ratio

# %%
print("=" * 70)
print("PART 1: Data Loading & Preprocessing")
print("=" * 70)

file_name = 'heloc_dataset_v1.csv'
dataset = pd.read_csv(file_name)

# Replace FICO special codes with 0
# -7: No usable/specific trade, -8: No bureau record, -9: No update
dataset.replace([-7, -8, -9], 0, inplace=True)

target = 'RiskPerformance'
if dataset[target].dtype == 'O':
    dataset[target] = dataset[target].map({'Bad': 0, 'Good': 1})

X = dataset.drop(target, axis=1)
y = dataset[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

print(f"Total samples     : {len(dataset):,}")
print(f"Training set      : {len(X_train):,}")
print(f"Test set          : {len(X_test):,}")
print(f"Target distribution (Total): Good={y.sum():,}, Bad={len(y)-y.sum():,}")
print(f"Target distribution (Test) : Good={y_test.sum():,}, Bad={len(y_test)-y_test.sum():,}")

# Descriptive statistics for key variables (Paper Table 3)
desc_vars = ['ExternalRiskEstimate', 'NetFractionRevolvingBurden',
             'NumSatisfactoryTrades', 'MSinceOldestTradeOpen']
desc_stats = X[desc_vars].describe().T[['mean', 'std', 'min', 'max']]
desc_stats.columns = ['Mean', 'Std', 'Min', 'Max']
print("\n[Descriptive Statistics - Key Variables]")
print(desc_stats.round(2))

# %% [markdown]
# # Part 2: XGBoost Model Training (Optuna Bayesian Optimization)

# %%
print("\n" + "=" * 70)
print("PART 2: XGBoost Hyperparameter Optimization (Optuna, 100 Trials)")
print("=" * 70)


def objective(trial):
    """Optuna objective function for XGBoost hyperparameter tuning."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'random_state': RANDOM_SEED,
        'eval_metric': 'logloss',
        'use_label_encoder': False
    }
    mdl = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(mdl, X_train, y_train, cv=cv, scoring='roc_auc')
    return scores.mean()


study = optuna.create_study(direction='maximize', study_name='xgb_credit')
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = study.best_params
best_params.update({
    'random_state': RANDOM_SEED,
    'eval_metric': 'logloss',
    'use_label_encoder': False
})

print(f"\nBest 5-Fold CV AUC: {study.best_value:.4f}")
print(f"Best Hyperparameters:\n{json.dumps(best_params, indent=2)}")

# Train final model with best hyperparameters
model = xgb.XGBClassifier(**best_params)
model.fit(X_train, y_train)

# %%
# --- Model Performance Evaluation (Paper Table 5) ---
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

perf_table = pd.DataFrame({
    'Metric': ['Accuracy', 'AUC-ROC', 'F1-Score', 'Precision', 'Recall',
               '5-Fold CV AUC (Train)'],
    'Value': [f"{acc:.4f}", f"{auc:.4f}", f"{f1:.4f}", f"{prec:.4f}",
              f"{rec:.4f}", f"{study.best_value:.4f}"]
})

print("\n" + "=" * 70)
print("[Paper Table 5] XGBoost Model Performance (Test Set)")
print("=" * 70)
print(perf_table.to_string(index=False))
print(f"\n[Confusion Matrix]  TN={cm[0][0]}  FP={cm[0][1]}")
print(f"                    FN={cm[1][0]}  TP={cm[1][1]}")

# Save results
perf_table.to_csv('model_performance.csv', index=False)

hp_table = pd.DataFrame([
    {'Parameter': k, 'Value': round(v, 6) if isinstance(v, float) else v}
    for k, v in best_params.items()
    if k not in ['random_state', 'eval_metric', 'use_label_encoder']
])
hp_table.to_csv('hyperparameters.csv', index=False)

# %% [markdown]
# # Part 3: Guardrail Configuration & Utility Functions

# %%
# --- Feature Classification (Paper Table 2) ---

# Immutable Features (Omega_imm):
#   Historical records and external scores that borrowers cannot modify
immutable_features = [
    'ExternalRiskEstimate',              # External credit score
    'MSinceOldestTradeOpen',             # Months since oldest trade
    'AverageMInFile',                    # Average months in file
    'MSinceMostRecentDelq',              # Months since most recent delinquency
    'MSinceMostRecentInqexcl7days',      # Months since most recent inquiry
    'NumBank2NatlTradesWHighUtilization'  # High utilization trades count
]

# Actionable Features (Omega_act):
#   Behavioral variables modifiable through short-term actions (within 6 months)
actionable_features = [
    'NetFractionRevolvingBurden',  # Credit card utilization ratio
    'NumSatisfactoryTrades',       # Number of satisfactory trades
    'NumTradesOpeninLast12M',      # New trades opened in last 12 months
    'PercentTradesNeverDelq',      # Percent of trades never delinquent
    'NumInqLast6M'                 # Number of inquiries in last 6 months
]

# All features treated as continuous
continuous_features = list(X.columns)

# --- DiCE Setup ---
d = dice_ml.Data(
    dataframe=dataset,
    continuous_features=continuous_features,
    outcome_name=target
)
m = dice_ml.Model(model=model, backend="sklearn")

# Primary search algorithm: Random method
# See Part 8 for algorithm comparison justification
exp_random = dice_ml.Dice(d, m, method="random")

# Identify rejected borrowers in test set
rejected_all = X_test[model.predict(X_test) == 0].copy()
print(f"Rejected borrowers in test set: {len(rejected_all):,}")
print(f"Counterfactual paths per sample: {TOTAL_CFS}")

# %%
# --- Utility Functions ---

def build_permitted_range(query_row, features, threshold):
    """
    Build permitted range for actionable features based on ±threshold.

    Implements Omega_act constraint (Section 3.2.3):
    Limits feature changes to within ±threshold of current value
    to prevent unrealistic short-term manipulations (gaming).

    Args:
        query_row: Single-row DataFrame of the query instance
        features: List of actionable feature names
        threshold: Allowable change ratio (e.g., 0.20 for ±20%)

    Returns:
        dict: {feature_name: [lower_bound, upper_bound]}
    """
    pr = {}
    for f in features:
        val = query_row[f].values[0]
        lo = min(val * (1 - threshold), val * (1 + threshold))
        hi = max(val * (1 - threshold), val * (1 + threshold))
        # Prevent zero-width ranges for near-zero values
        if abs(lo - hi) < 0.02:
            lo -= 0.5
            hi += 0.5
        pr[f] = [lo, hi]
    return pr


def check_causal_violations(orig_dict, cf_dict):
    """
    Check causal constraint (Omega_cau) violations (Section 3.2.2).

    Two causal rules are enforced:

    Rule 1 - Delinquency consistency:
        NumTrades90Ever2DerogPubRec <= NumTotalTrades * (1 - PercentTradesNeverDelq/100)
        Violation = derog count exceeds logical upper bound given never-delq rate

    Rule 2 - Revolving-Satisfactory direction consistency:
        If revolving burden decreases, satisfactory trades must not drop >10%
        Prevents window dressing via account closure strategy

    Args:
        orig_dict: Original instance feature values
        cf_dict: Counterfactual instance feature values

    Returns:
        int: Number of causal violations (0, 1, or 2)
    """
    violations = 0

    # Rule 1: Delinquency ratio vs derog count consistency
    ptndelq = cf_dict.get('PercentTradesNeverDelq',
                          orig_dict.get('PercentTradesNeverDelq', 0))
    derog = cf_dict.get('NumTrades90Ever2DerogPubRec',
                        orig_dict.get('NumTrades90Ever2DerogPubRec', 0))
    total = orig_dict.get('NumTotalTrades', 1)
    if total > 0:
        upper = total * (1 - ptndelq / 100.0)
        if derog > upper + 0.5:
            violations += 1

    # Rule 2: Revolving burden-satisfactory trades direction consistency
    rev_orig = orig_dict.get('NetFractionRevolvingBurden', 0)
    rev_cf = cf_dict.get('NetFractionRevolvingBurden', rev_orig)
    sat_orig = orig_dict.get('NumSatisfactoryTrades', 0)
    sat_cf = cf_dict.get('NumSatisfactoryTrades', sat_orig)
    if rev_cf < rev_orig and sat_cf < sat_orig * 0.9:
        violations += 1

    return violations


def analyze_cf_paths(query_row, cf_df, target_col, imm_feats, act_feats):
    """
    Analyze generated counterfactual paths at the path level.

    Evaluates each generated path for:
    - Success: Whether it achieves class flip (rejection -> approval)
    - Immutability violation: Whether immutable features were modified
    - Causal violation: Whether causal constraints are violated
    - Sparsity: Number of features changed in each successful path

    Args:
        query_row: Single-row DataFrame of original instance
        cf_df: DataFrame of generated counterfactual instances
        target_col: Name of target column
        imm_feats: List of immutable feature names
        act_feats: List of actionable feature names (None = check all)

    Returns:
        dict with path-level analysis results
    """
    result = {
        'total_paths': 0,
        'success_paths': 0,
        'imm_violation_paths': 0,
        'cau_violation_paths': 0,
        'feat_changes_list': []
    }

    if cf_df is None or cf_df.empty:
        return result

    orig = query_row.iloc[0]

    for i in range(len(cf_df)):
        cf_row = cf_df.iloc[i]

        if target_col not in cf_df.columns:
            continue
        pred = cf_row[target_col]
        if pd.isna(pred):
            continue

        result['total_paths'] += 1

        # Check if path achieves approval (class = 1)
        if int(pred) == 1:
            result['success_paths'] += 1

            # Check immutability violation (Omega_imm)
            imm_viol = any(
                abs(cf_row[f] - orig[f]) > 1e-5
                for f in imm_feats if f in cf_df.columns
            )
            if imm_viol:
                result['imm_violation_paths'] += 1

            # Check causal violation (Omega_cau)
            cau_viol = check_causal_violations(
                orig.to_dict(), cf_row.to_dict()
            )
            if cau_viol > 0:
                result['cau_violation_paths'] += 1

            # Count number of changed features (sparsity)
            check_feats = act_feats if act_feats else [
                c for c in X.columns if c in cf_df.columns
            ]
            n_changed = sum(
                abs(cf_row[f] - orig[f]) > 1e-5
                for f in check_feats if f in cf_df.columns
            )
            result['feat_changes_list'].append(n_changed)

    return result


# %% [markdown]
# # Part 4: Scenario Simulation (A / B / C)
#
# - **Scenario A (Vanilla)**: No constraints — all features can vary freely
# - **Scenario B (Immutability only)**: Immutable features are fixed ($\Omega_{imm}$)
# - **Scenario C (Full guardrails)**: Immutability + Causality + Actionability constraints

# %%
print("\n" + "=" * 70)
print("PART 4: Scenario Simulation (Full Test Set, 4 CFs/sample)")
print("=" * 70)


def run_scenario(scenario_name, rejected_df, exp_dice_obj, threshold=0.20):
    """
    Execute a guardrail scenario on all rejected borrowers.

    Metrics (Section 3.3 / 4.3):
        RR (Recourse Rate): % of samples with at least 1 successful path
        VR (Violation Rate): % of successful paths violating immutability
        Causal VR: % of successful paths violating causal constraints
        Reliable RR = RR × (1 - VR) × (1 - Causal VR)

    Args:
        scenario_name: "Scenario_A", "Scenario_B", or "Scenario_C"
        rejected_df: DataFrame of rejected borrowers
        exp_dice_obj: DiCE explainer object
        threshold: Actionability threshold for Scenario C
    """
    n = len(rejected_df)
    sample_success = 0
    total_paths = 0
    success_paths = 0
    imm_violations = 0
    cau_violations = 0
    feat_changes = []
    generated_paths = 0

    print(f"\n--- {scenario_name} (N={n}, CFs={TOTAL_CFS}/sample) ---")

    for i in tqdm(range(n), desc=scenario_name):
        query = rejected_df.iloc[i:i + 1]

        # Configure constraints based on scenario
        if scenario_name == "Scenario_A":
            ftv = "all"
            pr = None
        elif scenario_name == "Scenario_B":
            ftv = [c for c in X.columns if c not in immutable_features]
            pr = None
        elif scenario_name == "Scenario_C":
            ftv = actionable_features
            pr = build_permitted_range(query, actionable_features, threshold)

        try:
            result = exp_dice_obj.generate_counterfactuals(
                query,
                total_CFs=TOTAL_CFS,
                desired_class="opposite",
                features_to_vary=ftv,
                permitted_range=pr,
                proximity_weight=0.5,
                sparsity_weight=1.0,
                random_seed=RANDOM_SEED
            )
            cf_df = result.cf_examples_list[0].final_cfs_df

            act_f = actionable_features if scenario_name == "Scenario_C" else None
            pr_result = analyze_cf_paths(
                query, cf_df, target, immutable_features, act_f
            )

            generated_paths += pr_result['total_paths']
            success_paths += pr_result['success_paths']
            imm_violations += pr_result['imm_violation_paths']
            cau_violations += pr_result['cau_violation_paths']
            feat_changes.extend(pr_result['feat_changes_list'])

            if pr_result['success_paths'] > 0:
                sample_success += 1

        except Exception:
            continue

    # Compute metrics
    rr = sample_success / n if n > 0 else 0
    vr = imm_violations / success_paths if success_paths > 0 else 0
    cau_vr = cau_violations / success_paths if success_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cau_vr)
    avg_ch = np.mean(feat_changes) if feat_changes else 0

    row = {
        'Scenario': scenario_name,
        'N_Samples': n,
        'Sample_Success': sample_success,
        'RR(%)': round(rr * 100, 2),
        'Total_Paths_Generated': generated_paths,
        'Success_Paths': success_paths,
        'Imm_Violation_Paths': imm_violations,
        'VR(%)': round(vr * 100, 2),
        'Cau_Violation_Paths': cau_violations,
        'Causal_VR(%)': round(cau_vr * 100, 2),
        'Reliable_RR(%)': round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2)
    }

    print(f"  RR={row['RR(%)']:.2f}% | VR={row['VR(%)']:.2f}% | "
          f"Causal_VR={row['Causal_VR(%)']:.2f}%")
    print(f"  Reliable_RR={row['Reliable_RR(%)']:.2f}% | "
          f"Avg_Changed={row['Avg_Features_Changed']:.2f}")
    return row


# --- Run all three scenarios ---
start = time.time()
scenario_rows = []
for sc in ["Scenario_A", "Scenario_B", "Scenario_C"]:
    row = run_scenario(sc, rejected_all, exp_random, threshold=THRESHOLD)
    scenario_rows.append(row)

elapsed = time.time() - start
print(f"\n[Elapsed] {elapsed / 60:.1f} min")

df_scenarios = pd.DataFrame(scenario_rows)

# %%
# --- Paper Table 7: Guardrail Performance Comparison ---
print("\n" + "=" * 70)
print("[Paper Table 7] Guardrail Performance by Scenario (Path-Level)")
print("=" * 70)
cols = ['Scenario', 'N_Samples', 'RR(%)', 'Total_Paths_Generated',
        'Success_Paths', 'VR(%)', 'Causal_VR(%)',
        'Reliable_RR(%)', 'Avg_Features_Changed']
print(df_scenarios[cols].to_string(index=False))
df_scenarios.to_csv('scenario_comparison.csv', index=False)

# %% [markdown]
# # Part 5: Sensitivity Analysis (Threshold ±10% ~ ±30%)

# %%
print("\n" + "=" * 70)
print("PART 5: Sensitivity Analysis (Scenario C, varying threshold)")
print("=" * 70)

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30]
sensitivity_rows = []

for t in thresholds:
    label = f"±{int(t * 100)}%"
    print(f"\n--- Threshold: {label} ---")

    n = len(rejected_all)
    sample_success = 0
    total_paths = 0
    success_paths = 0
    imm_viol = 0
    cau_viol = 0
    feat_changes = []

    for i in tqdm(range(n), desc=label):
        query = rejected_all.iloc[i:i + 1]
        pr = build_permitted_range(query, actionable_features, t)

        try:
            result = exp_random.generate_counterfactuals(
                query,
                total_CFs=TOTAL_CFS,
                desired_class="opposite",
                features_to_vary=actionable_features,
                permitted_range=pr,
                proximity_weight=0.5,
                sparsity_weight=1.0,
                random_seed=RANDOM_SEED
            )
            cf_df = result.cf_examples_list[0].final_cfs_df
            pr_result = analyze_cf_paths(
                query, cf_df, target, immutable_features, actionable_features
            )

            total_paths += pr_result['total_paths']
            success_paths += pr_result['success_paths']
            imm_viol += pr_result['imm_violation_paths']
            cau_viol += pr_result['cau_violation_paths']
            feat_changes.extend(pr_result['feat_changes_list'])

            if pr_result['success_paths'] > 0:
                sample_success += 1
        except Exception:
            continue

    rr = sample_success / n if n > 0 else 0
    vr = imm_viol / success_paths if success_paths > 0 else 0
    cvr = cau_viol / success_paths if success_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cvr)
    avg_ch = np.mean(feat_changes) if feat_changes else 0

    sensitivity_rows.append({
        'Threshold': label, 'N': n,
        'RR(%)': round(rr * 100, 2),
        'VR(%)': round(vr * 100, 2),
        'Causal_VR(%)': round(cvr * 100, 2),
        'Reliable_RR(%)': round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2)
    })

df_sens = pd.DataFrame(sensitivity_rows)

# %%
# --- Paper Table 11: Sensitivity Analysis ---
print("\n" + "=" * 70)
print("[Paper Table 11] Sensitivity Analysis by Threshold")
print("=" * 70)
print(df_sens.to_string(index=False))
df_sens.to_csv('sensitivity_analysis.csv', index=False)

# %% [markdown]
# # Part 6: Subgroup Analysis (Credit Score Quartiles)

# %%
print("\n" + "=" * 70)
print("PART 6: Subgroup Analysis (ExternalRiskEstimate Quartiles)")
print("=" * 70)

ere = rejected_all['ExternalRiskEstimate']
q = ere.quantile([0.25, 0.5, 0.75])
q1, q2, q3 = q.iloc[0], q.iloc[1], q.iloc[2]
print(f"Quartile boundaries: Q1<={q1}, Q2<={q2}, Q3<={q3}")

subgroups = {
    'Q1 (Low)': rejected_all[ere <= q1],
    'Q2': rejected_all[(ere > q1) & (ere <= q2)],
    'Q3': rejected_all[(ere > q2) & (ere <= q3)],
    'Q4 (High)': rejected_all[ere > q3]
}

subgroup_rows = []

for grp_name, grp_df in subgroups.items():
    n_grp = len(grp_df)
    print(f"\n--- {grp_name} (N={n_grp}) ---")

    sample_success = 0
    success_paths = 0
    imm_viol = 0
    cau_viol = 0
    feat_changes = []

    for i in tqdm(range(n_grp), desc=grp_name):
        query = grp_df.iloc[i:i + 1]
        pr = build_permitted_range(query, actionable_features, THRESHOLD)

        try:
            result = exp_random.generate_counterfactuals(
                query,
                total_CFs=TOTAL_CFS,
                desired_class="opposite",
                features_to_vary=actionable_features,
                permitted_range=pr,
                proximity_weight=0.5,
                sparsity_weight=1.0,
                random_seed=RANDOM_SEED
            )
            cf_df = result.cf_examples_list[0].final_cfs_df
            pr_result = analyze_cf_paths(
                query, cf_df, target, immutable_features, actionable_features
            )

            success_paths += pr_result['success_paths']
            imm_viol += pr_result['imm_violation_paths']
            cau_viol += pr_result['cau_violation_paths']
            feat_changes.extend(pr_result['feat_changes_list'])

            if pr_result['success_paths'] > 0:
                sample_success += 1
        except Exception:
            continue

    rr = sample_success / n_grp if n_grp > 0 else 0
    vr = imm_viol / success_paths if success_paths > 0 else 0
    cvr = cau_viol / success_paths if success_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cvr)
    avg_ch = np.mean(feat_changes) if feat_changes else 0

    difficulty = ('Very High' if rrr < 0.05 else
                  'High' if rrr < 0.10 else
                  'Moderate' if rrr < 0.25 else 'Low')

    subgroup_rows.append({
        'Subgroup': grp_name, 'N': n_grp,
        'Sample_Success': sample_success,
        'RR(%)': round(rr * 100, 2),
        'Reliable_RR(%)': round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2),
        'Difficulty': difficulty
    })

df_sub = pd.DataFrame(subgroup_rows)

# %%
# --- Paper Table 12: Subgroup Analysis ---
print("\n" + "=" * 70)
print("[Paper Table 12] Recourse Difficulty by Credit Score Quartile")
print("=" * 70)
print(df_sub.to_string(index=False))
df_sub.to_csv('subgroup_analysis.csv', index=False)

# %% [markdown]
# # Part 7: Representative Case Extraction (Vanilla vs Proposed)
#
# Extracts cases where Vanilla DiCE violates immutable features
# while Proposed (Governed-DiCE) achieves approval through actionable changes only.
# (Paper Tables 8, 9, 10)

# %%
print("\n" + "=" * 70)
print("PART 7: Representative Case Extraction (Vanilla vs Proposed)")
print("=" * 70)

cases = []
for i in range(min(100, len(rejected_all))):
    query = rejected_all.iloc[i:i + 1]
    try:
        # Vanilla (no guardrails)
        v_res = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary="all", proximity_weight=0.5,
            random_seed=RANDOM_SEED
        )
        v_cf = v_res.cf_examples_list[0].final_cfs_df

        # Proposed (full guardrails)
        pr = build_permitted_range(query, actionable_features, THRESHOLD)
        p_res = exp_random.generate_counterfactuals(
            query, total_CFs=TOTAL_CFS, desired_class="opposite",
            features_to_vary=actionable_features,
            permitted_range=pr, proximity_weight=0.5,
            sparsity_weight=1.0, random_seed=RANDOM_SEED
        )
        p_cf = p_res.cf_examples_list[0].final_cfs_df

        if v_cf is None or v_cf.empty or p_cf is None or p_cf.empty:
            continue

        orig = query.iloc[0]

        # Find vanilla path with immutable violation + success
        for vi in range(len(v_cf)):
            v_row = v_cf.iloc[vi]
            if target in v_cf.columns and int(v_row[target]) == 1:
                has_viol = any(
                    abs(v_row[f] - orig[f]) > 1e-5
                    for f in immutable_features if f in v_cf.columns
                )
                if has_viol:
                    # Find proposed path with success
                    for pi in range(len(p_cf)):
                        p_row = p_cf.iloc[pi]
                        if target in p_cf.columns and int(p_row[target]) == 1:
                            cases.append({
                                'index': rejected_all.index[i],
                                'original': orig.to_dict(),
                                'vanilla_cf': v_row.to_dict(),
                                'proposed_cf': p_row.to_dict()
                            })
                            break
                    break
        if len(cases) >= 3:
            break
    except Exception:
        continue

# %%
# --- Display representative cases ---
if cases:
    key_vars = immutable_features[:3] + actionable_features[:3]
    for idx, case in enumerate(cases):
        print(f"\n[Case {idx + 1}] (Test Index: {case['index']})")
        print(f"{'Variable':<40} {'Current':>10} {'Vanilla CF':>12} {'Proposed CF':>12}")
        print("-" * 78)
        for v in key_vars:
            o = case['original'].get(v, 0)
            van = case['vanilla_cf'].get(v, 0)
            prop = case['proposed_cf'].get(v, 0)
            flag = " ⚠" if (v in immutable_features and
                            isinstance(van, (int, float)) and
                            abs(van - o) > 1e-5) else ""
            print(f"{v:<40} {o:>10.2f} {van:>12.2f} {prop:>12.2f}{flag}")
        print(f"{'[Result] Approval':<40} {'Reject':>10} {'Approve':>12} {'Approve':>12}")
else:
    print("No representative cases found. Try expanding the search range.")

# %% [markdown]
# # Part 8: Algorithm Robustness Validation (Random vs KD-Tree)
#
# Validates that the proposed guardrail framework operates independently
# of the search algorithm by comparing Random (stochastic) and
# KD-Tree (data-driven) methods under Scenario A and C.
#
# Note: Genetic method was excluded due to convergence failure
# in continuous high-dimensional settings (23 features, HELOC dataset).

# %%
print("\n" + "=" * 70)
print("PART 8: Algorithm Robustness Validation (Random vs KD-Tree)")
print("=" * 70)

METHODS = ["random", "kdtree"]


def run_scenario_method(scenario_name, method_name, exp_obj,
                        rejected_df, threshold=0.20):
    """
    Run a scenario with a specific DiCE search algorithm.

    Args:
        scenario_name: "Scenario_A" or "Scenario_C"
        method_name: "random" or "kdtree"
        exp_obj: DiCE explainer object for the given method
        rejected_df: DataFrame of rejected borrowers
        threshold: Actionability threshold
    """
    n = len(rejected_df)
    sample_success = 0
    success_paths = 0
    imm_violations = 0
    cau_violations = 0
    feat_changes = []
    generated_paths = 0
    errors = 0

    label = f"{scenario_name} × {method_name}"
    print(f"\n--- {label} (N={n}) ---")

    for i in tqdm(range(n), desc=label):
        query = rejected_df.iloc[i:i + 1]

        if scenario_name == "Scenario_A":
            ftv = "all"
            pr = None
        elif scenario_name == "Scenario_C":
            ftv = actionable_features
            pr = build_permitted_range(query, actionable_features, threshold)

        try:
            # random_seed is only supported by Random method
            cf_kwargs = dict(
                total_CFs=TOTAL_CFS,
                desired_class="opposite",
                features_to_vary=ftv,
                permitted_range=pr,
                proximity_weight=0.5,
                sparsity_weight=1.0,
            )
            if method_name == "random":
                cf_kwargs['random_seed'] = RANDOM_SEED

            dice_result = exp_obj.generate_counterfactuals(query, **cf_kwargs)
            cf_df = dice_result.cf_examples_list[0].final_cfs_df

            act_f = actionable_features if scenario_name == "Scenario_C" else None
            pr_result = analyze_cf_paths(
                query, cf_df, target, immutable_features, act_f
            )

            generated_paths += pr_result['total_paths']
            success_paths += pr_result['success_paths']
            imm_violations += pr_result['imm_violation_paths']
            cau_violations += pr_result['cau_violation_paths']
            feat_changes.extend(pr_result['feat_changes_list'])

            if pr_result['success_paths'] > 0:
                sample_success += 1

        except Exception:
            errors += 1
            continue

    rr = sample_success / n if n > 0 else 0
    vr = imm_violations / success_paths if success_paths > 0 else 0
    cvr = cau_violations / success_paths if success_paths > 0 else 0
    rrr = rr * (1 - vr) * (1 - cvr)
    avg_ch = np.mean(feat_changes) if feat_changes else 0

    row = {
        'Scenario': scenario_name, 'Method': method_name,
        'N': n, 'Sample_Success': sample_success,
        'RR(%)': round(rr * 100, 2),
        'Success_Paths': success_paths,
        'VR(%)': round(vr * 100, 2),
        'Causal_VR(%)': round(cvr * 100, 2),
        'Reliable_RR(%)': round(rrr * 100, 2),
        'Avg_Features_Changed': round(avg_ch, 2),
        'Errors': errors
    }

    print(f"  RR={row['RR(%)']:.2f}% | VR={row['VR(%)']:.2f}% | "
          f"Reliable_RR={row['Reliable_RR(%)']:.2f}% | Errors={errors}")
    return row


# --- Run algorithm comparison ---
algo_results = []
start = time.time()

for method in METHODS:
    print(f"\n{'=' * 50}")
    print(f"Method: {method.upper()}")
    print(f"{'=' * 50}")

    exp_obj = dice_ml.Dice(d, m, method=method)

    for scenario in ["Scenario_A", "Scenario_C"]:
        row = run_scenario_method(
            scenario, method, exp_obj, rejected_all, THRESHOLD
        )
        algo_results.append(row)

elapsed = time.time() - start
print(f"\n[Elapsed] {elapsed / 60:.1f} min")

df_algo = pd.DataFrame(algo_results)

# %%
# --- Paper Table X: Algorithm Robustness Comparison ---
print("\n" + "=" * 70)
print("[Paper Table X] Algorithm Robustness: Random vs KD-Tree")
print("=" * 70)
print(df_algo[['Scenario', 'Method', 'N', 'RR(%)', 'VR(%)',
               'Causal_VR(%)', 'Reliable_RR(%)',
               'Avg_Features_Changed', 'Errors']].to_string(index=False))
df_algo.to_csv('method_comparison.csv', index=False)

# --- Key findings summary ---
print("\n" + "=" * 70)
print("KEY FINDINGS")
print("=" * 70)

df_c = df_algo[df_algo['Scenario'] == 'Scenario_C']
vr_c = df_c['VR(%)'].tolist()
methods_c = df_c['Method'].tolist()

if all(v == 0.0 for v in vr_c):
    print("✓ Scenario C: VR = 0.00% for all algorithms")
    print("  → Immutability guardrail operates independently of search algorithm")

df_a = df_algo[df_algo['Scenario'] == 'Scenario_A']
print(f"\n✓ Scenario A Reliable RR:")
for _, r in df_a.iterrows():
    print(f"  {r['Method']}: {r['Reliable_RR(%)']:.2f}%")
print("  → No algorithm produces reliable recourse without guardrails")

# %% [markdown]
# # Summary
#
# ## Output Files
# | File | Description |
# |------|-------------|
# | `model_performance.csv` | XGBoost model performance metrics |
# | `hyperparameters.csv` | Optimal hyperparameters from Optuna |
# | `scenario_comparison.csv` | Scenario A/B/C comparison results |
# | `sensitivity_analysis.csv` | Threshold sensitivity analysis |
# | `subgroup_analysis.csv` | Credit score quartile subgroup analysis |
# | `method_comparison.csv` | Random vs KD-Tree algorithm comparison |

# %%
print("\n" + "=" * 70)
print(f"All experiments completed ({datetime.now().strftime('%Y-%m-%d %H:%M:%S')})")
print("=" * 70)
print(f"Configuration: total_CFs={TOTAL_CFS}, seed={RANDOM_SEED}, "
      f"threshold=±{int(THRESHOLD*100)}%")
print(f"Search algorithms: {METHODS}")
print(f"\nGenerated CSV files:")
print("  1. model_performance.csv    - Model performance metrics")
print("  2. hyperparameters.csv      - Optimal hyperparameters")
print("  3. scenario_comparison.csv  - Scenario A/B/C comparison")
print("  4. sensitivity_analysis.csv - Threshold sensitivity")
print("  5. subgroup_analysis.csv    - Credit quartile subgroups")
print("  6. method_comparison.csv    - Algorithm robustness (Random vs KD-Tree)")

C:\Users\ecredible\anaconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PART 1: Data Loading & Preprocessing
Total samples     : 10,459
Training set      : 8,367
Test set          : 2,092
Target distribution (Total): Good=5,000, Bad=5,459
Target distribution (Test) : Good=1,004, Bad=1,088

[Descriptive Statistics - Key Variables]
                              Mean     Std  Min    Max
ExternalRiskEstimate         67.94   19.28  0.0   94.0
NetFractionRevolvingBurden   32.28   29.27  0.0  232.0
NumSatisfactoryTrades        19.93   12.03  0.0   79.0
MSinceOldestTradeOpen       184.89  108.49  0.0  803.0

PART 2: XGBoost Hyperparameter Optimization (Optuna, 100 Trials)


Best trial: 74. Best value: 0.799972: 100%|██████████████████████████████████████████| 100/100 [05:58<00:00,  3.59s/it]



Best 5-Fold CV AUC: 0.8000
Best Hyperparameters:
{
  "n_estimators": 265,
  "max_depth": 5,
  "learning_rate": 0.013735894431469194,
  "subsample": 0.784997477773999,
  "colsample_bytree": 0.6226512140166448,
  "reg_alpha": 0.3458986965766233,
  "reg_lambda": 0.0038798023739030174,
  "min_child_weight": 10,
  "gamma": 0.00016821785874876824,
  "random_state": 42,
  "eval_metric": "logloss",
  "use_label_encoder": false
}

[Paper Table 5] XGBoost Model Performance (Test Set)
               Metric  Value
             Accuracy 0.6998
              AUC-ROC 0.7731
             F1-Score 0.6760
            Precision 0.7013
               Recall 0.6524
5-Fold CV AUC (Train) 0.8000

[Confusion Matrix]  TN=809  FP=279
                    FN=349  TP=655
Rejected borrowers in test set: 1,158
Counterfactual paths per sample: 4

PART 4: Scenario Simulation (Full Test Set, 4 CFs/sample)

--- Scenario_A (N=1158, CFs=4/sample) ---


Scenario_A: 100%|██████████████████████████████████████████████████████████████████| 1158/1158 [07:08<00:00,  2.70it/s]


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
  RR=11.05% | VR=0.00% | Reliable_RR=5.87% | Errors=1030

Method: KDTREE

--- Scenario_A × kdtree (N=1158) ---


Scenario_A × kdtree: 100%|█████████████████████████████████████████████████████████| 1158/1158 [12:38<00:00,  1.53it/s]


  RR=100.00% | VR=100.00% | Reliable_RR=0.00% | Errors=0

--- Scenario_C × kdtree (N=1158) ---


Scenario_C × kdtree: 100%|█████████████████████████████████████████████████████████| 1158/1158 [04:00<00:00,  4.81it/s]

  RR=0.00% | VR=0.00% | Reliable_RR=0.00% | Errors=1158

[Elapsed] 32.3 min

[Paper Table X] Algorithm Robustness: Random vs KD-Tree
  Scenario Method    N  RR(%)  VR(%)  Causal_VR(%)  Reliable_RR(%)  Avg_Features_Changed  Errors
Scenario_A random 1158 100.00  94.11         11.49            5.22                  2.77       0
Scenario_C random 1158  11.05   0.00         46.92            5.87                  1.99    1030
Scenario_A kdtree 1158 100.00 100.00         18.57            0.00                 15.93       0
Scenario_C kdtree 1158   0.00   0.00          0.00            0.00                  0.00    1158

KEY FINDINGS
✓ Scenario C: VR = 0.00% for all algorithms
  → Immutability guardrail operates independently of search algorithm

✓ Scenario A Reliable RR:
  random: 5.22%
  kdtree: 0.00%
  → No algorithm produces reliable recourse without guardrails

All experiments completed (2026-02-13 11:46:50)
Configuration: total_CFs=4, seed=42, threshold=±20%
Search algorithms: ['random', '

In [3]:
# %% [markdown]
# # Part 9: Detailed Case Analysis (Paper Tables 8, 9, 10)
#
# This section performs structured case-level comparison between:
# - **Tables 8 & 9**: Scenario A ("incidental pass") vs Scenario C ("engineered pass")
#   on identical borrowers — demonstrates qualitative difference behind similar Reliable RR
# - **Table 10**: Vanilla DiCE vs Governed-DiCE (Proposed) on a single borrower —
#   demonstrates how guardrails redirect from immutable manipulation to actionable paths
#
# Selection criteria for representative cases:
# 1. Both Scenario A and C must achieve approval (class flip to 1)
# 2. Scenario A path must rely on non-actionable variable changes
#    (large shifts in non-controllable features like NumTotalTrades, MaxDelq*)
# 3. Scenario C path must consist solely of small actionable adjustments (within ±20%)
# 4. For Table 10: Vanilla path must violate immutability constraints

# %%
print("\n" + "=" * 70)
print("PART 9: Detailed Case Analysis for Paper Tables 8, 9, 10")
print("=" * 70)

# ---------------------------------------------------------------------------
# 9.1  Helper: generate CF paths under a given scenario for a single borrower
# ---------------------------------------------------------------------------

def generate_single_cf(query, scenario, exp_obj, threshold=0.20):
    """
    Generate counterfactual paths for a single query under a given scenario.

    Args:
        query: Single-row DataFrame of the borrower
        scenario: "A" (no constraints), "B" (immutability only), or "C" (full)
        exp_obj: DiCE explainer object
        threshold: Actionability threshold for scenario C

    Returns:
        DataFrame of counterfactual paths, or None on failure
    """
    if scenario == "A":
        ftv = "all"
        pr = None
    elif scenario == "B":
        ftv = [c for c in X.columns if c not in immutable_features]
        pr = None
    elif scenario == "C":
        ftv = actionable_features
        pr = build_permitted_range(query, actionable_features, threshold)
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    try:
        res = exp_obj.generate_counterfactuals(
            query,
            total_CFs=TOTAL_CFS,
            desired_class="opposite",
            features_to_vary=ftv,
            permitted_range=pr,
            proximity_weight=0.5,
            sparsity_weight=1.0,
            random_seed=RANDOM_SEED
        )
        cf_df = res.cf_examples_list[0].final_cfs_df
        if cf_df is not None and not cf_df.empty:
            return cf_df
    except Exception:
        pass
    return None


# ---------------------------------------------------------------------------
# 9.2  Helper: compute per-feature delta table for a CF path
# ---------------------------------------------------------------------------

def build_delta_table(orig_series, cf_series, feature_list, classification):
    """
    Build a structured delta table comparing original vs counterfactual values.

    Args:
        orig_series: Original borrower feature values (pd.Series)
        cf_series: Counterfactual path feature values (pd.Series)
        feature_list: List of feature names to include
        classification: Feature classification dict {feature: category}

    Returns:
        DataFrame with columns: Variable, Category, Current, CF_Value,
                                Delta, Delta_Pct, Guardrail_Note
    """
    rows = []
    for f in feature_list:
        if f not in cf_series.index:
            continue
        o = orig_series[f]
        c = cf_series[f]
        delta = c - o
        pct = (delta / o * 100) if abs(o) > 1e-8 else (0 if abs(delta) < 1e-5 else float('inf'))
        cat = classification.get(f, 'Other')

        # Determine guardrail note
        if cat == 'Immutable':
            note = 'Fixed (historical record)' if abs(delta) < 1e-5 else '⚠ VIOLATION: immutable modified'
        elif cat == 'Actionable':
            if abs(delta) < 1e-5:
                note = 'No change'
            elif abs(pct) <= 20:
                note = f'Within ±20% threshold'
            else:
                note = f'Exceeds ±20% threshold'
        elif cat == 'Non-controllable':
            if abs(delta) < 1e-5:
                note = 'No change'
            else:
                note = f'Non-controllable variable modified'
        else:
            note = ''

        rows.append({
            'Variable': f,
            'Category': cat,
            'Current': round(o, 2),
            'CF_Value': round(c, 2),
            'Delta': round(delta, 2),
            'Delta_Pct': f"{pct:+.1f}%" if abs(pct) < 1e6 else 'N/A',
            'Guardrail_Note': note
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# 9.3  Feature classification mapping (extended from Part 3)
# ---------------------------------------------------------------------------

# Non-controllable features: not immutable but not short-term actionable
non_controllable_features = [
    'NumTotalTrades',
    'MaxDelq2PublicRecLast12M',
    'MaxDelqEver',
    'NumTrades90Ever2DerogPubRec',
    'NumTradesOpeninLast12M',
    'PercentInstallTrades',
    'MSinceMostRecentTradeOpen',
]

feature_classification = {}
for f in X.columns:
    if f in immutable_features:
        feature_classification[f] = 'Immutable'
    elif f in actionable_features:
        feature_classification[f] = 'Actionable'
    elif f in non_controllable_features:
        feature_classification[f] = 'Non-controllable'
    else:
        feature_classification[f] = 'Other'

# Priority features for display (immutable + actionable + key non-controllable)
display_features = (
    immutable_features +
    actionable_features +
    ['NumTotalTrades', 'MaxDelq2PublicRecLast12M',
     'NumTrades90Ever2DerogPubRec']
)
# Deduplicate while preserving order
display_features = list(dict.fromkeys(display_features))


# ---------------------------------------------------------------------------
# 9.4  Tables 8 & 9: Scenario A vs Scenario C on identical borrowers
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("9.4  Searching for Table 8 & 9 cases (Scenario A vs C comparison)")
print("-" * 70)

table89_cases = []
SEARCH_LIMIT = min(300, len(rejected_all))

for i in tqdm(range(SEARCH_LIMIT), desc="Scanning borrowers"):
    query = rejected_all.iloc[i:i + 1]
    orig = query.iloc[0]

    # Generate paths under both scenarios
    cf_a = generate_single_cf(query, "A", exp_random)
    cf_c = generate_single_cf(query, "C", exp_random)

    if cf_a is None or cf_c is None:
        continue

    # Find a successful Scenario A path with non-controllable variable changes
    best_a = None
    for j in range(len(cf_a)):
        row_a = cf_a.iloc[j]
        if target in cf_a.columns and int(row_a[target]) == 1:
            # Check for large non-controllable/non-actionable changes
            has_nonctrl_change = False
            for f in non_controllable_features:
                if f in cf_a.columns and abs(row_a[f] - orig[f]) > 1e-5:
                    pct = abs(row_a[f] - orig[f]) / max(abs(orig[f]), 1e-8) * 100
                    if pct > 50:  # Substantial change in non-controllable var
                        has_nonctrl_change = True
                        break
            if has_nonctrl_change:
                best_a = row_a
                break

    # Find a successful Scenario C path (should be actionable-only)
    best_c = None
    for j in range(len(cf_c)):
        row_c = cf_c.iloc[j]
        if target in cf_c.columns and int(row_c[target]) == 1:
            best_c = row_c
            break

    if best_a is not None and best_c is not None:
        # Verify Scenario C path uses only actionable features
        c_changes_only_actionable = True
        for f in X.columns:
            if f in actionable_features:
                continue
            if f in cf_c.columns and abs(best_c[f] - orig[f]) > 1e-5:
                c_changes_only_actionable = False
                break

        if c_changes_only_actionable:
            table89_cases.append({
                'test_index': rejected_all.index[i],
                'original': orig,
                'scenario_a_cf': best_a,
                'scenario_c_cf': best_c,
            })

    if len(table89_cases) >= 5:
        break

print(f"\nFound {len(table89_cases)} qualifying cases for Tables 8 & 9")

# %%
# --- Display Tables 8 & 9 cases ---
for idx, case in enumerate(table89_cases[:2]):
    table_num = 8 + idx
    test_id = case['test_index']
    orig = case['original']

    print(f"\n{'=' * 78}")
    print(f"[Paper Table {table_num}] Scenario A vs C — Borrower ID: {test_id}")
    print(f"{'=' * 78}")
    print(f"{'Category':<16} {'Variable':<38} {'Current':>8} {'Scen.A':>8} {'Scen.C':>8}  Guardrail Logic")
    print("-" * 110)

    for f in display_features:
        if f not in case['scenario_a_cf'].index:
            continue
        o = orig[f]
        a = case['scenario_a_cf'][f]
        c = case['scenario_c_cf'][f]
        cat = feature_classification.get(f, 'Other')

        # Only show rows with at least one change
        if abs(a - o) < 1e-5 and abs(c - o) < 1e-5:
            continue

        delta_a = a - o
        delta_c = c - o
        pct_a = (delta_a / o * 100) if abs(o) > 1e-8 else 0
        pct_c = (delta_c / o * 100) if abs(o) > 1e-8 else 0

        # Guardrail logic description
        if cat == 'Immutable':
            note = f"External score/time — borrower cannot modify"
        elif cat == 'Non-controllable':
            note = f"Non-actionable variable — short-term change infeasible"
        elif cat == 'Actionable':
            note = f"±20% range, actionable adjustment"
        else:
            note = ""

        a_str = f"{a:.1f}({pct_a:+.1f}%)" if abs(delta_a) > 1e-5 else f"{a:.1f}(fixed)"
        c_str = f"{c:.1f}({pct_c:+.1f}%)" if abs(delta_c) > 1e-5 else f"{c:.1f}(fixed)"

        print(f"{cat:<16} {f:<38} {o:>8.1f} {a_str:>8} {c_str:>8}  {note}")

    print(f"\n{'':>16} {'Approval':>38} {'Reject':>8} {'Approve':>8} {'Approve':>8}")
    print(f"{'':>16} {'Path character':>38} {'':>8} {'Incidental':>8} {'Engineered':>8}")


# ---------------------------------------------------------------------------
# 9.5  Table 10: Vanilla DiCE vs Governed-DiCE (Proposed)
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("9.5  Searching for Table 10 case (Vanilla vs Proposed)")
print("-" * 70)

table10_cases = []

for i in tqdm(range(SEARCH_LIMIT), desc="Scanning for Table 10"):
    query = rejected_all.iloc[i:i + 1]
    orig = query.iloc[0]

    # Vanilla = Scenario A (no constraints)
    cf_vanilla = generate_single_cf(query, "A", exp_random)
    # Proposed = Scenario C (full guardrails)
    cf_proposed = generate_single_cf(query, "C", exp_random)

    if cf_vanilla is None or cf_proposed is None:
        continue

    # Find vanilla path that violates immutability AND achieves approval
    best_van = None
    violated_imm_feats = []
    for j in range(len(cf_vanilla)):
        row_v = cf_vanilla.iloc[j]
        if target in cf_vanilla.columns and int(row_v[target]) == 1:
            v_imm = []
            for f in immutable_features:
                if f in cf_vanilla.columns and abs(row_v[f] - orig[f]) > 1e-5:
                    v_imm.append(f)
            if len(v_imm) >= 1:
                best_van = row_v
                violated_imm_feats = v_imm
                break

    # Find proposed path that succeeds with only actionable changes
    best_prop = None
    for j in range(len(cf_proposed)):
        row_p = cf_proposed.iloc[j]
        if target in cf_proposed.columns and int(row_p[target]) == 1:
            best_prop = row_p
            break

    if best_van is not None and best_prop is not None:
        table10_cases.append({
            'test_index': rejected_all.index[i],
            'original': orig,
            'vanilla_cf': best_van,
            'proposed_cf': best_prop,
            'violated_imm': violated_imm_feats
        })

    if len(table10_cases) >= 3:
        break

print(f"\nFound {len(table10_cases)} qualifying cases for Table 10")

# %%
# --- Display Table 10 ---
if table10_cases:
    case = table10_cases[0]
    test_id = case['test_index']
    orig = case['original']

    print(f"\n{'=' * 78}")
    print(f"[Paper Table 10] Vanilla DiCE vs Governed-DiCE — Borrower ID: {test_id}")
    print(f"{'=' * 78}")
    print(f"{'Category':<16} {'Variable':<38} {'Current':>8} {'Vanilla':>10} {'Proposed':>10}  Guardrail Logic")
    print("-" * 115)

    for f in display_features:
        if f not in case['vanilla_cf'].index:
            continue
        o = orig[f]
        v = case['vanilla_cf'][f]
        p = case['proposed_cf'][f]
        cat = feature_classification.get(f, 'Other')

        if abs(v - o) < 1e-5 and abs(p - o) < 1e-5:
            continue

        delta_v = v - o
        delta_p = p - o
        pct_v = (delta_v / o * 100) if abs(o) > 1e-8 else 0
        pct_p = (delta_p / o * 100) if abs(o) > 1e-8 else 0

        if cat == 'Immutable':
            if abs(delta_v) > 1e-5:
                note = "⚠ Immutable violation blocked by guardrail"
            else:
                note = "Fixed (historical record)"
        elif cat == 'Actionable':
            note = f"Actionable: ±20% threshold"
        elif cat == 'Non-controllable':
            note = "Non-controllable variable"
        else:
            note = ""

        v_str = f"{v:.1f}({pct_v:+.1f}%)" if abs(delta_v) > 1e-5 else f"{v:.1f}(fixed)"
        p_str = f"{p:.1f}({pct_p:+.1f}%)" if abs(delta_p) > 1e-5 else f"{p:.1f}(fixed)"

        print(f"{cat:<16} {f:<38} {o:>8.1f} {v_str:>10} {p_str:>10}  {note}")

    print(f"\n{'':>16} {'Approval':>38} {'Reject':>8} {'Approve':>10} {'Approve':>10}")
    print(f"\nViolated immutable features in Vanilla path: {case['violated_imm']}")


# ---------------------------------------------------------------------------
# 9.6  Export all case analysis data to CSV
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("9.6  Exporting case analysis data")
print("-" * 70)

export_rows = []

# Export Tables 8 & 9 cases
for idx, case in enumerate(table89_cases):
    orig = case['original']
    for f in X.columns:
        o = orig[f]
        a = case['scenario_a_cf'][f] if f in case['scenario_a_cf'].index else o
        c = case['scenario_c_cf'][f] if f in case['scenario_c_cf'].index else o
        export_rows.append({
            'Table': f'Table_{8 + idx}' if idx < 2 else f'Extra_Case_{idx + 1}',
            'Borrower_ID': case['test_index'],
            'Variable': f,
            'Category': feature_classification.get(f, 'Other'),
            'Current_Value': round(o, 4),
            'Scenario_A_CF': round(a, 4),
            'Scenario_C_CF': round(c, 4),
            'Delta_A': round(a - o, 4),
            'Delta_C': round(c - o, 4),
            'Pct_Change_A': round((a - o) / o * 100, 2) if abs(o) > 1e-8 else 0,
            'Pct_Change_C': round((c - o) / o * 100, 2) if abs(o) > 1e-8 else 0
        })

# Export Table 10 cases
for idx, case in enumerate(table10_cases):
    orig = case['original']
    for f in X.columns:
        o = orig[f]
        v = case['vanilla_cf'][f] if f in case['vanilla_cf'].index else o
        p = case['proposed_cf'][f] if f in case['proposed_cf'].index else o
        export_rows.append({
            'Table': f'Table_10_Case_{idx + 1}',
            'Borrower_ID': case['test_index'],
            'Variable': f,
            'Category': feature_classification.get(f, 'Other'),
            'Current_Value': round(o, 4),
            'Vanilla_CF': round(v, 4),
            'Proposed_CF': round(p, 4),
            'Delta_Vanilla': round(v - o, 4),
            'Delta_Proposed': round(p - o, 4),
            'Pct_Change_Vanilla': round((v - o) / o * 100, 2) if abs(o) > 1e-8 else 0,
            'Pct_Change_Proposed': round((p - o) / o * 100, 2) if abs(o) > 1e-8 else 0
        })

df_cases = pd.DataFrame(export_rows)
df_cases.to_csv('case_analysis_tables_8_9_10.csv', index=False)
print(f"Exported {len(export_rows)} rows to case_analysis_tables_8_9_10.csv")

# ---------------------------------------------------------------------------
# 9.7  Summary statistics for extracted cases
# ---------------------------------------------------------------------------
print("\n" + "-" * 70)
print("9.7  Case Summary")
print("-" * 70)

for idx, case in enumerate(table89_cases[:2]):
    orig = case['original']
    a_cf = case['scenario_a_cf']
    c_cf = case['scenario_c_cf']

    # Count changes per scenario
    a_changes = sum(1 for f in X.columns if f in a_cf.index and abs(a_cf[f] - orig[f]) > 1e-5)
    c_changes = sum(1 for f in X.columns if f in c_cf.index and abs(c_cf[f] - orig[f]) > 1e-5)

    # Count immutable violations in Scenario A
    a_imm_viols = sum(
        1 for f in immutable_features
        if f in a_cf.index and abs(a_cf[f] - orig[f]) > 1e-5
    )

    # Count non-controllable changes in Scenario A
    a_nonctrl = sum(
        1 for f in non_controllable_features
        if f in a_cf.index and abs(a_cf[f] - orig[f]) > 1e-5
    )

    print(f"\nTable {8 + idx} (Borrower {case['test_index']}):")
    print(f"  Scenario A: {a_changes} features changed "
          f"({a_imm_viols} immutable, {a_nonctrl} non-controllable)")
    print(f"  Scenario C: {c_changes} features changed (all actionable)")
    print(f"  → Qualitative difference: Incidental pass vs Engineered pass")

if table10_cases:
    case = table10_cases[0]
    orig = case['original']
    v_cf = case['vanilla_cf']
    p_cf = case['proposed_cf']
    v_changes = sum(1 for f in X.columns if f in v_cf.index and abs(v_cf[f] - orig[f]) > 1e-5)
    p_changes = sum(1 for f in X.columns if f in p_cf.index and abs(p_cf[f] - orig[f]) > 1e-5)
    print(f"\nTable 10 (Borrower {case['test_index']}):")
    print(f"  Vanilla:  {v_changes} features changed "
          f"(immutable violations: {case['violated_imm']})")
    print(f"  Proposed: {p_changes} features changed (actionable only)")

print("\n" + "=" * 70)
print("Part 9 complete. Case analysis exported to case_analysis_tables_8_9_10.csv")
print("=" * 70)


PART 9: Detailed Case Analysis for Paper Tables 8, 9, 10

----------------------------------------------------------------------
9.4  Searching for Table 8 & 9 cases (Scenario A vs C comparison)
----------------------------------------------------------------------


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.04s/it]

Scanning borrowers:   0%|▏                                                             | 1/300 [00:01<07:34,  1.52s/it]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.45it/s]

Scanning borrowers:   1%|▍                                                             | 2/300 [00:02<05:42,  1.15s/it]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec


...

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.08it/s]

Scanning for Table 10:   4%|██▌                                                       | 13/300 [00:11<04:00,  1.19it/s]

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec



100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.94it/s]

Scanning for Table 10:   4%|██▌                                                       | 13/300 [00:11<04:18,  1.11it/s]


Found 3 qualifying cases for Table 10

[Paper Table 10] Vanilla DiCE vs Governed-DiCE — Borrower ID: 2145
Category         Variable                                Current    Vanilla   Proposed  Guardrail Logic
-------------------------------------------------------------------------------------------------------------------
Immutable        MSinceOldestTradeOpen                     190.0 782.0(+311.6%) 190.0(fixed)  ⚠ Immutable violation blocked by guardrail
Actionable       NetFractionRevolvingBurden                 41.0 41.0(fixed) 39.0(-4.9%)  Actionable: ±20% threshold
Actionable       NumSatisfactoryTrades                      37.0 37.0(fixed) 44.0(+18.9%)  Actionable: ±20% threshold

                                               Approval   Reject    Approve    Approve

Violated immutable features in Vanilla path: ['MSinceOldestTradeOpen']

----------------------------------------------------------------------
9.6  Exporting case analysis data
-----------------------------------